[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/moving_charge/moving_charge.ipynb)

# Moving charges

Two opposite charges at rest side by side have the electric field of a dipole. Sped up towards the speed of light in opposite directions, each is the same charge seen through its own boost: its field flattens into a disc across its motion and gains a magnetic part that circles its path. The field is never written down here. The potential's gradient is a map from a small step in spacetime to the change in potential; the gradients of the charges add, and contracting the sum against an open vector gives the field. Contracting the field's own gradient gives Maxwell's equations, and so does a trace that needs no metric. A charge circling at constant speed radiates, and the same contraction of its retarded potential shows the waves.

In [1]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext
from numga.algebras import STA as ga
from examples.electromagnetism.moving_charge import render
from examples.electromagnetism.moving_charge.core import Charge, Orbit, grid

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Antivector = ga.gatype.antivector()
Even = ga.gatype.even()
Odd = ga.gatype.odd()
PotentialGradient = ga.gatype((Vector, Vector))                             # Vector <- Vector
FieldGradient = ga.gatype((Bivector, Vector))                               # Bivector <- Vector

# Two opposite charges side by side, one entry per charge: their strengths, where they are at lab
# time zero, and the sense of their motion along x; and one core for both.
strengths = np.array([1.0, -1.0])
positions = mv.y * np.array([0.35, -0.35])                                  # [charges] Vector
directions = np.array([1.0, -1.0])
core_radius = 0.08
# Two rapidities, one entry per case: at rest, and moving at about 0.96 of the speed of light.
rapidities = np.array([0.0, 2.0])
# The lab at time zero: a window in the plane z = 0, sampled at the centres of its cells.
half_width, half_height = 1.5, 1.0
columns, rows = 300, 200

## 1. The potential and its gradient

At rest, the potential of a charge points along its time direction `mv.t` and falls off as one over the distance, softened inside a core so that it stays finite. Its change over a small step in spacetime is a map, `Vector <- Vector`, written in closed form. A moving charge is the same charge seen through a boost, a rotor in the plane `xt`: the map at rest, with the step pulled into the rest frame by `boost << Vector` and the change pushed back out by `boost >> ...`. The motion enters nowhere else. Potentials add, and so do their gradients: the charges are one more batch axis, summed.

In [ ]:
def profile(charges: Charge, events: Vector) -> tuple[Vector, Scalar, Scalar]:
    """The spatial separation from each charge in its rest frame, the field per unit separation, and
    that weight's change per unit of `separation | step`."""
    # `boost << ...` undoes the boost: each event, from the charge's position, as the charge's rest
    # frame sees it. Its spatial part there is what is left after removing the part along the rest time.
    at_rest = charges.boost << (events - charges.position)                    # [...] Vector
    separations = at_rest - (at_rest | mv.t) * mv.t                           # [...] Vector
    # Space squares to minus one, so minus the square is the squared distance, softened by the core.
    softened = charges.core_radius**2 - (separations | separations)           # [...] Scalar
    # Coulomb's law per unit separation: the field's strength over the distance, so that `weight`
    # times a separation is the field itself. Beyond the core it falls off as the inverse cube.
    weight = charges.charge / (4 * np.pi) / (softened * softened.square_root())   # [...] Scalar
    return separations, weight, 3 * weight / softened


def potential_gradient(charges: Charge, events: Vector) -> PotentialGradient:
    """The change in each charge's potential for a small step from each event."""
    separations, weight, _ = profile(charges, events)                         # [...] Vector, Scalar
    # At rest the potential points along the rest time and changes with the distance: a bare
    # `Vector` leaves the step open, so this is a map from a step to a change in potential.
    at_rest = weight * mv.t * (separations | Vector)                          # [...] Vector <- Vector
    # The step pulled into the rest frame, and the change pushed back out to the lab: the same map,
    # seen through the boost.
    return charges.boost >> at_rest(charges.boost << Vector)                  # [...] Vector <- Vector


# A boost by a rapidity is a rotor in the plane of x and time; at rapidity zero, the identity. Each
# charge moves in its own sense along x.
boosts = (mv.xt * (rapidities[:, None] * directions / 2)).exp()             # [cases, charges] Rotor
# Two new axes broadcast each case over the window; the charges stay on the last axis.
charges = Charge(strengths, core_radius, boosts[:, None, None, :], positions)
events = grid(half_width, half_height, columns, rows)                         # [rows, columns] Vector
# The gradients of the two charges add: a sum over the last axis.
potential_gradients = potential_gradient(charges, events[..., None]).sum(axis=-1)   # [cases, rows, columns] Vector <- Vector

## 2. The field, by contracting the gradient

Multiplying an open vector into the gradient's output and pairing it with the gradient's input through the metric, `(Vector * gradient(Vector)).contract(1, 2)`, gives the derivative of the potential. Its scalar part vanishes: the potential keeps the Lorenz gauge. Its bivector part is the field, which the wedge picks out alone, `(Vector ^ gradient(Vector)).contract(1, 2)`. The lab splits the field along its own time: the field's part along `mv.t` is the electric field, its dual's part the magnetic field. Below, at rest (top) the field is purely electric, the field of a dipole; moving (bottom) each charge's field flattens into a disc across its motion, and magnetic fields circle the paths, crossing the plane in opposite senses on either side of each.

In the tensor formulation of electrodynamics the bivector part reads as $F^{\mu\nu} = \partial^\mu A^\nu - \partial^\nu A^\mu$, and the vanishing scalar part as the Lorenz condition $\partial_\mu A^\mu = 0$.

In [ ]:
def potential_derivative(gradients: PotentialGradient) -> Even:
    # `gradients(Vector)` keeps the step open, and multiplying an open vector into the output opens a
    # second slot: a map from two vectors to an even element. `contract` pairs those two slots through
    # the metric, summing each basis vector times the gradient of its reciprocal, without naming a basis.
    return (Vector * gradients(Vector)).contract(1, 2)                        # [...] Even


def field(gradients: PotentialGradient) -> Bivector:
    # The same contraction with the wedge in place of the product keeps the bivector part alone.
    return (Vector ^ gradients(Vector)).contract(1, 2)                        # [...] Bivector


fields = field(potential_gradients)                                           # [cases, rows, columns] Bivector
render.fields(events, fields);

## 3. Maxwell's equations, with and without the metric

The field's own gradient is a map `Bivector <- Vector`, and contracting it the same way gives the derivative of the field: a vector, the current, and a trivector that vanishes. That one statement is Maxwell's equations. They also follow without the metric. The regressive product `&` pairs the gradient's output with an open antivector, and tracing the antivector against the gradient's input, `(gradient(Vector) & Antivector).trace(1, 2)`, pairs a vector with an antivector by incidence alone. On the field this gives the current; on the field's dual it gives the magnetic source, which vanishes. The metric enters only through that dual. The current's part along the lab's time is the charge density, shown below close up, positive red and negative blue: round at rest, flattened along the motion and denser when moving.

In the tensor formulation of electrodynamics the two traces read as $\partial_\mu F^{\mu\nu} = J^\nu$ and $\partial_\mu \tilde F^{\mu\nu} = 0$.

In [ ]:
def field_gradient(charges: Charge, events: Vector) -> FieldGradient:
    """The change in each charge's field for a small step from each event."""
    separations, weight, weight_change = profile(charges, events)            # [...] Vector, Scalar, Scalar
    # At rest the field is `weight * (separations ^ mv.t)`: radial, and purely electric. A step
    # changes its strength by its part along the separation...
    radial = weight_change * (separations ^ mv.t) * (separations | Vector)    # [...] Bivector <- Vector
    # ...and moves the point the field is radial from, turning it.
    turning = weight * (Vector ^ mv.t)                                        # [...] Bivector <- Vector
    # Seen through the boost, as for the potential.
    return charges.boost >> (radial + turning)(charges.boost << Vector)       # [...] Bivector <- Vector


def field_derivative(gradients: FieldGradient) -> Odd:
    # The same contraction as for the potential: a vector part, the current, and a trivector part,
    # which vanishes.
    return (Vector * gradients(Vector)).contract(1, 2)                        # [...] Odd


def sources(gradients: FieldGradient) -> tuple[Vector, Vector]:
    # The output met with an open antivector leaves a scalar map of two slots, a vector and an
    # antivector; tracing them pairs each basis vector with the antivector it meets in one. No metric
    # is involved: only which blades meet.
    current = (gradients(Vector) & Antivector).trace(1, 2)                    # [...] Vector
    # The dual is where the metric enters; the same trace on it gives the magnetic source.
    magnetic_source = (gradients(Vector).dual() & Antivector).trace(1, 2)     # [...] Vector
    return current, magnetic_source


# A close window around the charges, where their density lives; their field gradients add.
close_half_width, close_half_height = 0.6, 0.5
close = grid(close_half_width, close_half_height, columns, rows)              # [rows, columns] Vector
field_gradients = field_gradient(charges, close[..., None]).sum(axis=-1)      # [cases, rows, columns] Bivector <- Vector
field_derivatives = field_derivative(field_gradients)                         # [cases, rows, columns] Odd
currents, magnetic_sources = sources(field_gradients)                         # [cases, rows, columns] Vector each
render.densities(close, currents);

## 4. Speeding up

The charges sped up from rest to the top rapidity and slowed down again, every stage at once: the rapidities are one more batch axis. The outlines mark the charges, where their density is high.

In [ ]:
# The rapidity rising and falling over the frames, on a coarser window, and each frame's duration.
sweep_frames = 60
sweep_rapidities = rapidities[-1] * (1 - np.cos(2 * np.pi * np.arange(sweep_frames) / sweep_frames)) / 2
sweep_columns, sweep_rows = 150, 100
frame_ms = 60

# One pair of boosts per frame, in place of one per case: the functions above run unchanged, once,
# over all frames together.
sweep_boosts = (mv.xt * (sweep_rapidities[:, None] * directions / 2)).exp()  # [frames, charges] Rotor
sweep_charges = Charge(strengths, core_radius, sweep_boosts[:, None, None, :], positions)
sweep_events = grid(half_width, half_height, sweep_columns, sweep_rows)       # [rows, columns] Vector
sweep_fields = field(potential_gradient(sweep_charges, sweep_events[..., None]).sum(axis=-1))   # [frames, rows, columns] Bivector
sweep_currents, _ = sources(field_gradient(sweep_charges, sweep_events[..., None]).sum(axis=-1))   # [frames, rows, columns] Vector
render.inline(render.sweep(sweep_events, sweep_fields, sweep_currents), frame_ms)

## 5. A circling charge radiates

A point charge circling at constant speed: its potential at an event is set by where the charge was when it sent the light that reaches the event, at the retarded time. Each event finds that time by a few steps back along the light cone, which always close in, and then Newton's steps. The gradient of the potential is again a closed-form map, built from the charge's velocity and acceleration and the ray of light, and contracting it gives the field: the bound field that moves with the charge, and the waves it sheds. Shown is the magnetic field at one moment, times the distance from the centre so that the waves keep their strength across the picture; as the charge circles, the whole pattern turns with it. At half the speed of light (left) the charge sheds a smooth spiral; at 0.9 (right) the waves bunch into sharp pulses thrown forward, a beam that sweeps round with the charge.

In the potential formulation of electrodynamics this potential reads as the Liénard–Wiechert potential $A^\mu = \dfrac{q}{4\pi}\dfrac{u^\mu}{u_\nu R^\nu}$, evaluated at the retarded time.

In [ ]:
def worldline(orbit: Orbit, times: Scalar) -> tuple[Vector, Vector, Vector]:
    """The circling charge at each lab time, its change per unit lab time, and that change's change."""
    turning = orbit.speed / orbit.radius                                      # angular rate
    # A turn of the starting offset by the angle covered: a rotor in the orbit's plane.
    offset = (mv.xy * (-0.5 * turning) * times).exp() >> (mv.x * orbit.radius)   # [...] Vector
    # Per unit lab time the charge moves one unit along time, and along the offset turned a quarter
    # turn in the plane; that motion turns in its turn, towards the centre.
    tangent = mv.t + turning * (offset | mv.xy)                               # [...] Vector
    bend = turning * ((tangent - mv.t) | mv.xy)                               # [...] Vector
    return times * mv.t + offset, tangent, bend


def retarded(orbit: Orbit, events: Vector, settle_steps: int, newton_steps: int) -> Scalar:
    """The lab time at which the charge sent the light that reaches each event."""
    def light(times: Scalar) -> tuple[Scalar, Scalar, Scalar]:
        """How far light from the charge at each time falls short of the event, its rate, and the distance."""
        position, tangent, _ = worldline(orbit, times)                        # [...] Vector each
        # The ray's spatial part in the lab, and its length.
        across = (events - position) - ((events - position) | mv.t) * mv.t   # [...] Vector
        distance = (-(across | across)).square_root()                         # [...] Scalar
        return (events | mv.t) - times - distance, 1 + (across | tangent) / distance, distance

    # Back by the distance light covers, again and again: slower than light, the charge cannot keep
    # up, so the steps always close in. Then Newton's steps finish the job.
    times = events | mv.t                                                     # [...] Scalar
    for _ in range(settle_steps):
        _, _, distance = light(times)
        times = (events | mv.t) - distance
    for _ in range(newton_steps):
        shortfall, rate, _ = light(times)
        times = times + shortfall / rate
    return times


def orbit_potential_gradient(orbit: Orbit, events: Vector) -> PotentialGradient:
    """The change in the circling charge's potential for a small step from each event."""
    emitted = retarded(orbit, events, settle_steps, newton_steps)             # [...] Scalar
    position, tangent, bend = worldline(orbit, emitted)                       # [...] Vector each
    # The charge's velocity and acceleration in its own time, and the ray of light from it to the event.
    dilation = 1 / np.sqrt(1 - orbit.speed**2)
    velocity, acceleration = dilation * tangent, dilation**2 * bend           # [...] Vector each
    ray = events - position                                                   # [...] Vector
    # The distance as the charge saw it when it sent the light: the ray measured along its velocity.
    reach = ray | velocity                                                    # [...] Scalar
    # The potential is `weight * velocity`. A step moves the emission along the ray, which changes
    # the velocity and the reach; the bare `Vector`s keep the step open.
    emission = (ray | Vector) / reach                                         # [...] Scalar <- Vector
    reach_change = (velocity | Vector) + ((ray | acceleration) - 1) * emission   # [...] Scalar <- Vector
    weight = orbit.charge / (4 * np.pi) / reach                               # [...] Scalar
    return weight * (acceleration * emission - velocity * reach_change / reach)   # [...] Vector <- Vector


# One charge on a circle: its strength and the circle's radius, at two speeds, one entry per case;
# and the retarded time's steps.
orbit_strength, orbit_radius = 1.0, 1.0
speeds = np.array([0.5, 0.9])
settle_steps, newton_steps = 25, 6
orbit = Orbit(orbit_strength, orbit_radius, speeds[:, None, None])
# Lab time zero on a wide window of the orbit's plane, fine enough for the fast charge's thin pulses.
orbit_half_width = 10.0
orbit_samples = 300
plane = grid(orbit_half_width, orbit_half_width, orbit_samples, orbit_samples)   # [rows, columns] Vector
# Lab time zero once more, to place the charge itself.
now = mv.scalar([0.0])                                                         # [1] Scalar

# The field by the same contraction as for the charges in straight lines.
radiated = field(orbit_potential_gradient(orbit, plane))                       # [cases, rows, columns] Bivector
places, _, _ = worldline(orbit, now)                                           # [cases, 1, 1] Vector
render.radiation(plane, radiated, places);

In [ ]:
# checks
# The potential keeps the Lorenz gauge: its derivative is the field alone.
np.testing.assert_allclose((potential_derivative(potential_gradients) - fields).kernel, 0.0, atol=1e-12)
# The metric contraction and the metric-free trace give the same current, and no magnetic source.
np.testing.assert_allclose((field_derivatives - currents).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose(magnetic_sources.kernel, 0.0, atol=1e-12)
# The current is each charge's softened density carried along its own time, summed.
at_rest = charges.boost << (close[..., None] - charges.position)
separations = at_rest - (at_rest | mv.t) * mv.t
softened = core_radius**2 - (separations | separations)
charge_density = 3 * strengths * core_radius**2 / (4 * np.pi) / (softened * softened * softened.square_root())
carried = (charge_density * (charges.boost >> mv.t)).sum(axis=-1)
np.testing.assert_allclose((currents - carried).kernel, 0.0, atol=1e-9)
# The light from the circling charge at the retarded time reaches each event.
emitted_at = plane + mv.t * 3.0                                               # [rows, columns] Vector
emitter, _, _ = worldline(orbit, retarded(orbit, emitted_at, settle_steps, newton_steps))
rays = emitted_at - emitter                                                   # [cases, rows, columns] Vector
np.testing.assert_allclose((rays | rays).kernel, 0.0, atol=1e-10)